In [7]:
#
print("\033[1m Team Member: Isaac\033[0m")
print()
print("\033[1mTask: Yahoo Finance + market metrics + financial statements \033[0m")
print()
#

from functools import partial

from src.data_sources.yahoo_finance import YahooFinanceClient
from src.tools.financial_tools import get_cash_flow, get_financials
from src.tools.market_tools import get_company_info, get_price_data


yahoo_client = YahooFinanceClient()

print("Yahoo Finance tools loaded")


In [8]:
#
print("\033[1m Team Member: Isaac\033[0m")
print()
print("\033[1mTask: Yahoo Finance + market metrics + financial statements \033[0m")
print()
#

TOOLS = {
    "price_data": partial(get_price_data, yahoo_client),
    "company_info": partial(get_company_info, yahoo_client),
    "financials": partial(get_financials, yahoo_client),
    "cash_flow": partial(get_cash_flow, yahoo_client),
}

print("Available tools:")

for name in TOOLS:
    print(" -", name)


Available tools:
 - price_data
 - company_info
 - financials
 - cash_flow


In [9]:
#
print("\033[1m Team Member: Christina\033[0m")
print()
print("\033[1mTask: Memory integration + persistent memory \033[0m")
print()
#


def load_memory():
    if not MEMORY_FILE.exists():
        return []

    try:
        with open(MEMORY_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return []


def save_memory(symbol, summary, weaknesses):
    memories = load_memory()

    memories.append({
        "date": datetime.now().isoformat(),
        "symbol": symbol,
        "summary": summary,
        "weaknesses": weaknesses,
    })

    # Keep the most recent 20 memories
    memories = memories[-20:]

    with open(MEMORY_FILE, "w", encoding="utf-8") as f:
        json.dump(memories, f, indent=2)


def get_memory(symbol):
    memories = load_memory()

    return [
        m for m in memories
        if m.get("symbol") == symbol
    ]


In [13]:
#
print("\033[1m Team Members: Brandon, Christina and Isaac\033[0m")
print()
print("\033[1mTask: Planner Agent + Market Research Agent  \033[0m")
print()
#


import json
import re
from numbers import Number

from src.llm import ask_llm


class MarketResearchAgent:
    # =========================================================
    # LLM
    # =========================================================

    def ask(self, prompt):
        return ask_llm(prompt)

    # =========================================================
    # JSON PARSER
    # =========================================================

    def parse_json_response(self, response, fallback=None):
        """
        Safely parse JSON returned by an LLM.

        Handles:
        - normal JSON
        - Markdown code fences
        - explanatory text before/after JSON
        """

        if not response or not isinstance(response, str):
            return fallback

        text = response.strip()

        # Attempt 1: direct JSON
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            pass

        # Remove Markdown code fences
        text = re.sub(
            r"^```(?:json)?\s*",
            "",
            text,
            flags=re.IGNORECASE,
        )

        text = re.sub(
            r"\s*```$",
            "",
            text,
        ).strip()

        # Attempt 2: JSON after removing fences
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            pass

        # Attempt 3: extract JSON object
        start = text.find("{")
        end = text.rfind("}")

        if start != -1 and end > start:
            candidate = text[start:end + 1]

            try:
                return json.loads(candidate)
            except json.JSONDecodeError:
                pass

        return fallback

    # =========================================================
    # PLAN
    # =========================================================

    def plan(self, symbol):
        previous_memory = get_memory(symbol)

        if previous_memory is None:
            previous_memory = []

        if not isinstance(previous_memory, list):
            previous_memory = [previous_memory]

        recent_memory = previous_memory[-3:]
        available_tools = list(TOOLS.keys())

        prompt = f"""
You are the planning component of a financial research agent.

Stock symbol:
{symbol}

Previous research memory:
{json.dumps(recent_memory, indent=2, default=str)}

Available tools:
{json.dumps(available_tools, indent=2)}

Create a research plan.

Rules:

1. Only select tools from the available tool list.
2. Do not invent tools.
3. Prefer tools that directly answer the research objectives.
4. Use previous memory to avoid repeating unresolved work.
5. Do not make investment recommendations.
6. Do not assume data exists before the tools are executed.

Return ONLY a valid JSON object.

Required schema:

{{
    "objectives": [],
    "tools": [],
    "questions": []
}}
"""

        response = self.ask(prompt)

        fallback = {
            "objectives": [
                "Analyze price performance",
                "Analyze company information",
                "Analyze valuation",
                "Analyze financial performance",
                "Analyze cash generation",
            ],
            "tools": [
                tool
                for tool in [
                    "price_data",
                    "company_info",
                    "financials",
                    "cash_flow",
                ]
                if tool in TOOLS
            ],
            "questions": [
                "How has the stock performed?",
                "What is the company's business and operating profile?",
                "What valuation metrics are available?",
                "How are revenue and earnings performing?",
                "What does the available cash-flow data show?",
            ],
        }

        plan = self.parse_json_response(response, fallback)

        if not isinstance(plan, dict):
            plan = fallback

        objectives = plan.get("objectives", [])
        tools = plan.get("tools", [])
        questions = plan.get("questions", [])

        if not isinstance(objectives, list):
            objectives = []

        if not isinstance(tools, list):
            tools = []

        if not isinstance(questions, list):
            questions = []

        # Only allow actual tools
        tools = [
            tool
            for tool in tools
            if isinstance(tool, str) and tool in TOOLS
        ]

        if not tools:
            tools = fallback["tools"]

        return {
            "objectives": objectives,
            "tools": tools,
            "questions": questions,
        }

    # =========================================================
    # TOOL EXECUTION
    # =========================================================

    def execute_tools(self, symbol, plan):
        observations = {}

        for tool_name in plan.get("tools", []):
            print(f"🔧 Running tool: {tool_name}")

            if tool_name not in TOOLS:
                observations[tool_name] = {
                    "error": "Tool is not available."
                }
                continue

            try:
                tool = TOOLS[tool_name]
                result = tool(symbol)

                if result is None:
                    result = {
                        "error": "Tool returned no data."
                    }

                observations[tool_name] = result

            except Exception as exc:
                observations[tool_name] = {
                    "error": f"{type(exc).__name__}: {exc}"
                }

        return observations

    # =========================================================
    # DATA VALIDATION
    # =========================================================

    def validate_observations(self, symbol, observations):
        issues = []

        def find_value(obj, possible_keys):
            if isinstance(obj, dict):
                for key in possible_keys:
                    if key in obj:
                        return obj[key]

                for value in obj.values():
                    result = find_value(value, possible_keys)

                    if result is not None:
                        return result

            elif isinstance(obj, list):
                for item in obj:
                    result = find_value(item, possible_keys)

                    if result is not None:
                        return result

            return None

        # -----------------------------------------------------
        # Tool-level validation
        # -----------------------------------------------------

        for tool_name, data in observations.items():
            if data is None:
                issues.append({
                    "type": "missing",
                    "tool": tool_name,
                    "message": "Tool returned None.",
                })
                continue

            if isinstance(data, dict) and "error" in data:
                issues.append({
                    "type": "tool_error",
                    "tool": tool_name,
                    "message": str(data["error"]),
                })

        company = observations.get("company_info", {})
        financials = observations.get("financials", {})
        cash_flow = observations.get("cash_flow", {})
        price = observations.get("price_data", {})

        # -----------------------------------------------------
        # Locate important values
        # -----------------------------------------------------

        market_cap = find_value(
            company,
            [
                "market_cap",
                "marketCap",
                "marketCapitalization",
            ],
        )

        revenue = find_value(
            financials,
            [
                "total_revenue",
                "totalRevenue",
                "revenue",
                "Total Revenue",
            ],
        )

        operating_cash_flow = find_value(
            cash_flow,
            [
                "operating_cash_flow",
                "operatingCashFlow",
                "operating_cashflow",
                "Operating Cash Flow",
            ],
        )

        free_cash_flow = find_value(
            cash_flow,
            [
                "free_cash_flow",
                "freeCashFlow",
                "Free Cash Flow",
            ],
        )

        latest_price = find_value(
            price,
            [
                "latest_price",
                "current_price",
                "currentPrice",
                "regularMarketPrice",
                "price",
                "Current Price",
            ],
        )

        one_year_return = find_value(
            price,
            [
                "one_year_return",
                "one_year_return_pct",
                "1y_return",
                "return_1y",
            ],
        )

        # -----------------------------------------------------
        # Cash-flow consistency
        # -----------------------------------------------------

        if operating_cash_flow is not None and free_cash_flow is None:
            issues.append({
                "type": "missing",
                "field": "free_cash_flow",
                "message": (
                    "Operating cash flow exists, but free cash flow "
                    "is not available."
                ),
            })

        # -----------------------------------------------------
        # Price validation
        # -----------------------------------------------------

        if latest_price is None:
            issues.append({
                "type": "missing",
                "field": "latest_price",
                "message": "Latest stock price was not supplied.",
            })

        # -----------------------------------------------------
        # Return validation
        # -----------------------------------------------------

        if one_year_return is None:
            issues.append({
                "type": "missing",
                "field": "one_year_return",
                "message": "One-year return was not supplied.",
            })

        # -----------------------------------------------------
        # Market-cap sanity check
        # -----------------------------------------------------

        if (
            isinstance(market_cap, Number)
            and isinstance(revenue, Number)
            and revenue > 0
        ):
            ratio = market_cap / revenue

            if ratio > 1000:
                issues.append({
                    "type": "suspicious",
                    "field": "market_cap",
                    "message": (
                        "Market capitalization is more than 1000x "
                        "reported revenue. Verify units and scaling "
                        "before using this value."
                    ),
                    "market_cap": market_cap,
                    "revenue": revenue,
                    "ratio": ratio,
                })

        # -----------------------------------------------------
        # Percentage sanity checks
        # -----------------------------------------------------

        def check_percent_field(data, keys):
            value = find_value(data, keys)

            if isinstance(value, Number) and abs(value) > 1000:
                issues.append({
                    "type": "suspicious",
                    "field": keys[0],
                    "message": (
                        "Percentage-like value is unusually large. "
                        "Verify units."
                    ),
                    "value": value,
                })

        check_percent_field(
            price,
            [
                "one_year_return",
                "one_year_return_pct",
            ],
        )

        check_percent_field(
            company,
            [
                "profit_margin",
                "operating_margin",
                "dividend_yield",
                "profitMargins",
                "operatingMargins",
                "dividendYield",
            ],
        )

        return issues

    # =========================================================
    # REFLECTION
    # =========================================================

    def reflect(
        self,
        symbol,
        plan,
        observations,
        validation_issues,
    ):
        prompt = f"""
You are the data-quality auditor of a financial research agent.

Stock:
{symbol}

Research plan:
{json.dumps(plan, indent=2, default=str)}

Collected data:
{json.dumps(observations, indent=2, default=str)}

Automated validation findings:
{json.dumps(validation_issues, indent=2, default=str)}

Your task is to inspect the supplied data carefully.

Rules:

1. Inspect the actual supplied fields.
2. Never claim a value is missing if it exists anywhere in the data.
3. Distinguish missing information from suspicious information.
4. Do not invent values.
5. Do not infer future values.
6. Check units such as dollars, thousands, millions, billions,
   percentages, and ratios.
7. Check whether financial figures appear internally consistent.
8. Treat suspicious values as requiring verification, not automatically
   as incorrect.
9. Do not ask questions unrelated to the research.
10. Do not provide investment advice.
11. Do not evaluate whether the stock should be bought or sold.

Return ONLY valid JSON:

{{
    "strengths": [],
    "weaknesses": [],
    "missing_information": [],
    "suspicious_values": [],
    "follow_up_questions": []
}}
"""

        response = self.ask(prompt)

        fallback = {
            "strengths": [],
            "weaknesses": [],
            "missing_information": [],
            "suspicious_values": [],
            "follow_up_questions": [],
        }

        reflection = self.parse_json_response(response, fallback)

        if not isinstance(reflection, dict):
            return fallback

        fields = [
            "strengths",
            "weaknesses",
            "missing_information",
            "suspicious_values",
            "follow_up_questions",
        ]

        for field in fields:
            if not isinstance(reflection.get(field), list):
                reflection[field] = []

        # Add deterministic validation findings
        for issue in validation_issues:
            issue_type = issue.get("type")
            message = issue.get("message")

            if issue_type == "missing":
                if message not in reflection["missing_information"]:
                    reflection["missing_information"].append(message)

            elif issue_type == "suspicious":
                if message not in reflection["suspicious_values"]:
                    reflection["suspicious_values"].append(message)

            elif issue_type == "tool_error":
                if message not in reflection["weaknesses"]:
                    reflection["weaknesses"].append(message)

        return reflection

    # =========================================================
    # REPORT
    # =========================================================

    def report(
        self,
        symbol,
        plan,
        observations,
        reflection,
        validation_issues,
    ):
        prompt = f"""
You are a neutral financial research reporting system.

Create a factual research report for {symbol}.

Research plan:
{json.dumps(plan, indent=2, default=str)}

Collected tool observations:
{json.dumps(observations, indent=2, default=str)}

Automated validation:
{json.dumps(validation_issues, indent=2, default=str)}

Quality review:
{json.dumps(reflection, indent=2, default=str)}

Structure:

# {symbol} Market Research

## 1. Company Overview

## 2. Price Performance

## 3. Valuation

## 4. Financial Performance

## 5. Cash Flow

## 6. Risks and Uncertainties

## 7. Data Quality

## 8. Further Research

STRICT RULES:

- Use ONLY information supplied in the observations.
- Do not use outside knowledge.
- Do not invent missing values.
- Do not estimate missing values.
- Do not silently correct suspicious values.
- Preserve the units supplied by the tools.
- Every numerical claim must be traceable to supplied data.
- If a number has been flagged as suspicious, explicitly identify it
  as requiring verification.
- Do not claim information is missing when it exists in the observations.
- Distinguish factual observations from interpretation.
- Do not calculate correlations.
- Do not make predictions unless the supplied data explicitly contains
  a prediction.
- Do not provide personalized investment advice.
- Do not say buy, sell, or hold.
- Do not give an overall investment rating.

Important:

If the observations contain operating cash flow or free cash flow,
the Cash Flow section MUST discuss those values.

If the latest stock price is missing, say that it is unavailable.

If a valuation number looks suspicious, report it as supplied and
clearly flag the unit/scaling issue rather than replacing it.

Return the report as Markdown.
"""

        return self.ask(prompt)

    # =========================================================
    # REPORT VALIDATION
    # =========================================================

    def validate_report(
        self,
        symbol,
        report,
        observations,
    ):
        issues = []

        if not report or not isinstance(report, str):
            return [
                "Report generation returned an empty or invalid response."
            ]

        report_lower = report.lower()

        cash_flow = observations.get("cash_flow", {})

        def contains_key_recursive(obj, keys):
            if isinstance(obj, dict):
                for key, value in obj.items():
                    if key in keys:
                        return True

                    if contains_key_recursive(value, keys):
                        return True

            elif isinstance(obj, list):
                for item in obj:
                    if contains_key_recursive(item, keys):
                        return True

            return False

        has_cash_data = contains_key_recursive(
            cash_flow,
            {
                "operating_cash_flow",
                "operatingCashFlow",
                "Operating Cash Flow",
                "free_cash_flow",
                "freeCashFlow",
                "Free Cash Flow",
            },
        )

        if has_cash_data:
            missing_phrases = [
                "cash generation data is missing",
                "cash flow data is missing",
                "cash generation is missing",
            ]

            for phrase in missing_phrases:
                if phrase in report_lower:
                    issues.append(
                        "Report incorrectly claims cash-flow data is missing."
                    )

        prohibited_phrases = [
            "buy the stock",
            "sell the stock",
            "you should buy",
            "you should sell",
            "strong buy",
            "strong sell",
        ]

        for phrase in prohibited_phrases:
            if phrase in report_lower:
                issues.append(
                    f"Report contains unsupported investment language: {phrase}"
                )

        return issues

    # =========================================================
    # MEMORY
    # =========================================================

    def remember(
        self,
        symbol,
        report,
        reflection,
    ):
        prompt = f"""
Summarize this financial research for future research sessions.

Report:
{report}

Quality review:
{json.dumps(reflection, indent=2, default=str)}

Create 3-5 concise notes.

Focus on:

- important factual findings
- unresolved research questions
- suspicious or questionable data
- data limitations
- useful future research

Rules:

- Do not invent information.
- Do not provide investment advice.
- Do not say buy, sell, or hold.
- Keep each note concise.

Return plain text.
"""

        summary = self.ask(prompt)

        if not summary:
            summary = "No research summary was generated."

        save_memory(
            symbol,
            summary,
            reflection.get("weaknesses", []),
        )

    # =========================================================
    # FORMATTERS
    # =========================================================

    @staticmethod
    def format_billions(value):
        if not isinstance(value, Number):
            return "N/A"

        return f"${value / 1_000_000_000:.3f}B"

    @staticmethod
    def format_trillions(value):
        if not isinstance(value, Number):
            return "N/A"

        return f"${value / 1_000_000_000_000:.3f}T"

    @staticmethod
    def format_number(value, decimals=2):
        if not isinstance(value, Number):
            return "N/A"

        return f"{value:,.{decimals}f}"

    @staticmethod
    def format_percent(value, decimals=2):
        if not isinstance(value, Number):
            return "N/A"

        return f"{value * 100:.{decimals}f}%"

    @staticmethod
    def format_ratio(a, b):
        if (
            not isinstance(a, Number)
            or not isinstance(b, Number)
            or b == 0
        ):
            return "N/A"

        return f"{a / b:.3f}x"

    @staticmethod
    def get_latest_year(data):
        """
        Safely determine the newest year/key from a financial dictionary.
        """

        if not isinstance(data, dict) or not data:
            return None

        keys = list(data.keys())

        year_keys = []

        for key in keys:
            match = re.search(r"(19|20)\d{2}", str(key))

            if match:
                year_keys.append((match.group(0), key))

        if year_keys:
            year_keys.sort(
                key=lambda item: item[0],
                reverse=True,
            )
            return year_keys[0][1]

        return keys[0]

    @staticmethod
    def get_nested(data, key, default=None):
        if not isinstance(data, dict):
            return default

        return data.get(key, default)

    # =========================================================
    # CRISP FINAL REPORT
    # =========================================================

    def print_final_report(self, research_result):
        symbol = research_result.get(
            "symbol",
            "UNKNOWN",
        )

        observations = research_result.get(
            "observations",
            {},
        )

        info = observations.get(
            "company_info",
            {},
        )

        financials = observations.get(
            "financials",
            {},
        )

        cash_flow = observations.get(
            "cash_flow",
            {},
        )

        validation = research_result.get(
            "validation",
            [],
        )

        # -----------------------------------------------------
        # Latest financial year
        # -----------------------------------------------------

        latest_year = self.get_latest_year(financials)

        if latest_year is None:
            latest_year = self.get_latest_year(cash_flow)

        # -----------------------------------------------------
        # Financial values
        # -----------------------------------------------------

        revenue = None
        operating_income = None
        net_income = None
        ebitda = None
        eps = None

        if latest_year:
            revenue = self.get_nested(
                financials.get("Total Revenue", {}),
                latest_year,
            )

            operating_income = self.get_nested(
                financials.get("Operating Income", {}),
                latest_year,
            )

            net_income = self.get_nested(
                financials.get("Net Income", {}),
                latest_year,
            )

            ebitda = self.get_nested(
                financials.get("EBITDA", {}),
                latest_year,
            )

            eps = self.get_nested(
                financials.get("Diluted EPS", {}),
                latest_year,
            )

        # -----------------------------------------------------
        # Cash flow
        # -----------------------------------------------------

        operating_cf = None
        free_cf = None
        capex = None

        if latest_year:
            operating_cf = self.get_nested(
                cash_flow.get("Operating Cash Flow", {}),
                latest_year,
            )

            free_cf = self.get_nested(
                cash_flow.get("Free Cash Flow", {}),
                latest_year,
            )

            capex = self.get_nested(
                cash_flow.get("Capital Expenditure", {}),
                latest_year,
            )

        # -----------------------------------------------------
        # Company fields
        # -----------------------------------------------------

        long_name = info.get(
            "longName",
            symbol,
        )

        sector = info.get(
            "sector",
            "N/A",
        )

        industry = info.get(
            "industry",
            "N/A",
        )

        country = info.get(
            "country",
            "N/A",
        )

        current_price = info.get("currentPrice")
        market_cap = info.get("marketCap")
        enterprise_value = info.get("enterpriseValue")

        trailing_pe = info.get("trailingPE")
        forward_pe = info.get("forwardPE")

        price_to_sales = info.get(
            "priceToSalesTrailing12Months"
        )

        profit_margin = info.get("profitMargins")
        operating_margin = info.get("operatingMargins")
        roe = info.get("returnOnEquity")
        beta = info.get("beta")
        dividend_yield = info.get("dividendYield")

        # -----------------------------------------------------
        # Print
        # -----------------------------------------------------

        print("=" * 70)
        print("FINAL RESEARCH REPORT")
        print("=" * 70)

        print(f"\n{long_name} ({symbol})")
        print("-" * 70)

        print("\n1. COMPANY OVERVIEW")

        print(f"   Sector:              {sector}")
        print(f"   Industry:            {industry}")
        print(f"   Country:             {country}")

        print(
            f"   Current Price:       "
            f"${self.format_number(current_price)}"
        )

        print(
            f"   Market Cap:          "
            f"{self.format_trillions(market_cap)}"
        )

        print(
            f"   Enterprise Value:    "
            f"{self.format_trillions(enterprise_value)}"
        )

        print("\n2. VALUATION")

        print(
            f"   Trailing P/E:        "
            f"{self.format_number(trailing_pe)}"
        )

        print(
            f"   Forward P/E:         "
            f"{self.format_number(forward_pe)}"
        )

        print(
            f"   Price/Sales (TTM):   "
            f"{self.format_number(price_to_sales)}"
        )

        print(
            f"   Profit Margin:       "
            f"{self.format_percent(profit_margin)}"
        )

        print(
            f"   Operating Margin:    "
            f"{self.format_percent(operating_margin)}"
        )

        print(
            f"   Return on Equity:    "
            f"{self.format_percent(roe)}"
        )

        print(
            f"   Beta:                "
            f"{self.format_number(beta, 3)}"
        )

        year_label = (
            str(latest_year)[:4]
            if latest_year
            else "N/A"
        )

        print(
            f"\n3. FINANCIAL PERFORMANCE ({year_label})"
        )

        print(
            f"   Revenue:             "
            f"{self.format_billions(revenue)}"
        )

        print(
            f"   Operating Income:    "
            f"{self.format_billions(operating_income)}"
        )

        print(
            f"   Net Income:          "
            f"{self.format_billions(net_income)}"
        )

        print(
            f"   EBITDA:              "
            f"{self.format_billions(ebitda)}"
        )

        print(
            f"   Diluted EPS:         "
            f"${self.format_number(eps)}"
        )

        print("\n4. EBITDA / NET INCOME ANALYSIS")

        print(
            f"   EBITDA:              "
            f"{self.format_billions(ebitda)}"
        )

        print(
            f"   Net Income:          "
            f"{self.format_billions(net_income)}"
        )

        print(
            f"   EBITDA / Net Income: "
            f"{self.format_ratio(ebitda, net_income)}"
        )

        print(f"\n5. CASH FLOW ({year_label})")

        print(
            f"   Operating Cash Flow: "
            f"{self.format_billions(operating_cf)}"
        )

        print(
            f"   Free Cash Flow:      "
            f"{self.format_billions(free_cf)}"
        )

        print(
            f"   Capital Expenditure: "
            f"{self.format_billions(capex)}"
        )

        print("\n6. DIVIDEND")

        print(
            f"   Dividend Yield:      "
            f"{self.format_percent(dividend_yield)}"
        )

        print("\n7. DATA VALIDATION")

        if validation:
            for item in validation:
                field = item.get(
                    "field",
                    "general",
                )

                message = item.get(
                    "message",
                    str(item),
                )

                print(
                    f"   - {field}: {message}"
                )
        else:
            print("   No validation issues reported.")

        missing = [
            item.get("field", "unknown")
            for item in validation
            if item.get("type") == "missing"
        ]

        print("\n8. MISSING INFORMATION")

        if missing:
            for field in missing:
                print(f"   - {field}")
        else:
            print("   None")

        print("\n" + "=" * 70)
        print("END OF FINAL RESEARCH REPORT")
        print("=" * 70)

    # =========================================================
    # MAIN RUNNER
    # =========================================================

    def run(self, symbol):
        symbol = symbol.upper().strip()

        if not symbol:
            raise ValueError(
                "Stock symbol cannot be empty."
            )

        print("=" * 60)
        print(f"AGENTIC MARKET RESEARCH: {symbol}")
        print("=" * 60)

        # =====================================================
        # 1. PLAN
        # =====================================================

        print("\n🧠 STEP 1 — PLANNING")

        plan = self.plan(symbol)

        print(
            json.dumps(
                plan,
                indent=2,
                default=str,
            )
        )

        # =====================================================
        # 2. TOOL EXECUTION
        # =====================================================

        print("\n🔧 STEP 2 — DYNAMIC TOOL USE")

        observations = self.execute_tools(
            symbol,
            plan,
        )

        print("\n📊 RAW OBSERVATIONS")

        print(
            json.dumps(
                observations,
                indent=2,
                default=str,
            )
        )
                
        # =====================================================
        # 3. DATA VALIDATION
        # =====================================================

        print("\n🛡️ STEP 3 — DATA VALIDATION")

        validation_issues = self.validate_observations(
            symbol,
            observations,
        )

        if validation_issues:
            print(
                json.dumps(
                    validation_issues,
                    indent=2,
                    default=str,
                )
            )
        else:
            print("No automated validation issues found.")

        # =====================================================
        # 4. SELF REFLECTION
        # =====================================================

        print("\n🔍 STEP 4 — SELF-REFLECTION")

        reflection = self.reflect(
            symbol,
            plan,
            observations,
            validation_issues,
        )

        print(
            json.dumps(
                reflection,
                indent=2,
                default=str,
            )
        )

       # =====================================================
       # 5. REPORT
       # =====================================================

        print("\n📝 STEP 5 — REPORT GENERATION")

        report = self.report(
            symbol,
            plan,
            observations,
            reflection,
            validation_issues,
       )

        print("\n--- GENERATED REPORT ---")

        if report:
             print(report)
        else:
            print("❌ REPORT IS EMPTY")

            print("--- END GENERATED REPORT ---")


        # =====================================================
        # 6. REPORT VALIDATION
        # =====================================================

        print("\n🔎 STEP 6 — REPORT VALIDATION")

        report_issues = self.validate_report(
            symbol,
            report,
            observations,
        )

        if report_issues:
            for issue in report_issues:
                print(f"⚠️ {issue}")
        else:
            print("Report passed validation.")

        # =====================================================
        # 7. MEMORY
        # =====================================================

        print("\n💾 STEP 7 — LEARNING / MEMORY")

        self.remember(
            symbol,
            report,
            reflection,
        )

        print("Memory updated.")

        # =====================================================
        # STRUCTURED RESULT
        # =====================================================

        research_result = {
            "symbol": symbol,
            "plan": plan,
            "observations": observations,
            "validation": validation_issues,
            "reflection": reflection,
            "report": report,
            "report_validation": report_issues,
        }

        # =====================================================
        # CRISP CONSOLE REPORT
        # =====================================================

        self.print_final_report(
            research_result
        )

        return research_result


# =============================================================
# CREATE AGENT
# =============================================================

agent = MarketResearchAgent()

print("Agent created successfully")


# =============================================================
# MAIN PROGRAM
# =============================================================

def main():
    ticker = input(
        "Enter stock ticker (e.g., AAPL, MSFT, NVDA): "
    ).strip().upper()

    if not ticker:
        print("❌ No ticker entered.")
        return

    try:
        # -----------------------------------------------------
        # Verify ticker before starting agent
        # -----------------------------------------------------

        print(f"\n🔎 Checking ticker: {ticker}")

        history = yahoo_client.get_history(ticker, period="5d")

        if not history:
            print(
                f"❌ Could not find market data for '{ticker}'."
            )
            return

        print(
            f"✅ Found {ticker}. Starting research...\n"
        )

        # -----------------------------------------------------
        # Run agent
        # -----------------------------------------------------

        research_result = agent.run(ticker)

        # -----------------------------------------------------
        # Display LLM-generated Markdown report
        # -----------------------------------------------------

        print("\n" + "=" * 70)
        print("LLM RESEARCH REPORT")
        print("=" * 70)

        print(
            research_result.get(
                "report",
                "No report was generated.",
            )
        )

    except Exception as e:
        print(
            f"❌ Error checking or researching "
            f"ticker '{ticker}': {type(e).__name__}: {e}"
        )


# =============================================================
# ENTRY POINT
# =============================================================

if __name__ == "__main__":
    main()


 Team Members: Brandon, Christina and Isaac

Task: Planner Agent + Market Research Agent  

Agent created successfully


Enter stock ticker (e.g., AAPL, MSFT, NVDA):  NVDA



🔎 Checking ticker: NVDA
✅ Found NVDA. Starting research...

AGENTIC MARKET RESEARCH: NVDA

🧠 STEP 1 — PLANNING
{
  "objectives": [
    "Verify the accuracy of the capital expenditure data for 2026",
    "Analyze the reasons behind the negative capital expenditure for 2026",
    "Investigate the implications of the high valuation metrics on NVIDIA's financial performance"
  ],
  "tools": [
    "price_data",
    "company_info",
    "financials",
    "cash_flow"
  ],
  "questions": [
    "cash_flow.getcapital_expenditure(2026, '2026-01-31')",
    "cash_flow.getoperating_cash_flow(2026)",
    "financials.getEBITDAandNetIncomeRatio(2026)",
    "price_data.getstockPrice()",
    "company_info.get latest stock price and one-year return"
  ]
}

🔧 STEP 2 — DYNAMIC TOOL USE
🔧 Running tool: price_data
🔧 Running tool: company_info
🔧 Running tool: financials
🔧 Running tool: cash_flow

📊 RAW OBSERVATIONS
{
  "price_data": {
    "start_price": 176.25,
    "latest_price": 222.27,
    "1y_return_percen